## Generate a fund timeseries PowerPoint report

Reads one saved check under `output-data/timeseries-checks/<ISIN>/` (from `check-fund-timeseries.ipynb`) and writes `outputs/fund-timeseries-report.pptx` in that folder.

Optional inputs consumed when present: `outputs/worst-mismatch-summary.json`, `outputs/llm-evaluation-openai.json`.

The layout comes from `helpers/timeseries_functions/assets/fund_timeseries_report_template.pptx`. Edit it in PowerPoint: text may contain `{tokens}` (see `report_values` in `fund_timeseries_report.py`), and the shapes named `chart_returns` and `chart_regression` mark where the charts go.

In [1]:
import json
import sys
from pathlib import Path


def _ensure_helpers_importable() -> None:
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "helpers").is_dir():
            root = str(directory)
            if root not in sys.path:
                sys.path.insert(0, root)
            return
    raise FileNotFoundError(f"Could not find helpers/, starting from {start}")


_ensure_helpers_importable()

from helpers.timeseries_functions.fund_llm_dossier import timeseries_check_dir
from helpers.timeseries_functions.fund_timeseries_report import (
    load_report_data,
    write_fund_timeseries_report,
)

SELECTED_FUND = "US4613088016"

In [2]:
check_dir = timeseries_check_dir(SELECTED_FUND)
data = load_report_data(check_dir)

print(f"Loaded check folder: {check_dir}")
print(f"Fund: {data.fund_static['name']}")
print(f"ISIN: {data.fund_static['isin']}")
print(f"Regression: {data.regression['equation']}")
print(
    f"Source agreement: "
    f"{data.comparison_summary['perfect_matches'] + data.comparison_summary['acceptable_matches']}"
    f" / {data.comparison_summary['n_months']} months"
)

preview = {
    "parameters": data.parameters,
    "comparison_summary": data.comparison_summary,
    "regression_statistics": data.regression["statistics"],
    "has_llm_evaluation": data.llm_evaluation is not None,
}
print(json.dumps(preview, indent=2))

Loaded check folder: /Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/output-data/timeseries-checks/US4613088016
Fund: American Funds Investment Co of America;529-E
ISIN: US4613088016
Regression: Lipper = -0.06% + 0.927 × S&P 500 TR
Source agreement: 200 / 202 months
{
  "parameters": {
    "master_timeseries": "Lipper",
    "comparison_timeseries": "Yahoo",
    "index_column": "S&P 500 TR",
    "master_column": "US4613088016 (Lipper)",
    "comparison_column": "US4613088016 (Yahoo)",
    "match_tolerance": 0.0001,
    "spike_return": 0.1,
    "reversal_band": 0.15,
    "scale_factor": 5,
    "scale_window": 12,
    "level_jump": 0.5,
    "index_gap": 0.05,
    "date_gap_days": 45
  },
  "comparison_summary": {
    "master_timeseries": "Lipper",
    "comparison_timeseries": "Yahoo",
    "master_column": "US4613088016 (Lipper)",
    "comparison_column": "US4613088016 (Yahoo)",
    "match_tolerance": 0.0001,
    "n_months": 202,
    "perfect_matches": 1,
    "ac

In [3]:
report_path = write_fund_timeseries_report(check_dir)
print(f"Saved report: {report_path}")

Saved report: /Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/output-data/timeseries-checks/US4613088016/outputs/fund-timeseries-report.pptx
